# ⚡ Universal Hardware-Accelerated WebGPU Chrome on Google Colab (Tesla T4)
### پلتفرم ابری اجرای Google Chrome با شتاب کامل سخت‌افزاری WebGPU و Vulkan روی تسلا T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hosein-ul/colab-webgpu-chrome/blob/main/colab_chrome_webgpu.ipynb)

این پلتفرم به یک پروژه خاص محدود نیست؛ بلکه یک محیط کامل **Google Chrome ابری با شتاب سخت‌افزاری کارت گرافیک NVIDIA Tesla T4 (۱۵ گیگابایت VRAM)** است که می‌توانید هرگونه وب‌اپلیکیشن سنگین، شیدرهای محاسباتی WebGPU، بنچمارک‌های سه‌بعدی و افزونه‌های مرورگر (مانند ولت‌ها) را با شتاب سخت‌افزاری کامل اجرا نمایید.

---
### ⚡ انتخاب روش دسترسی:
* **روش اول (موتور مدرن KasmVNC - ۱۰۰٪ تک‌کلیک و بدون اکانت):** استریم فوق‌العاده سبک با فشرده‌سازی داینامیک WebP، بدون کوچک‌ترین لگ در نشانگر موس (Client-side cursor) و بدون نیاز به کپی کردن هیچ کدی.
* **روش دوم (Google Chrome Remote Desktop - استریم ویدئویی ۶۰ فریم):** برای کارهای فوق‌سنگین و طولانی با پروتکل WebRTC و تاخیر ناچیز زیر ۳۰ میلی‌ثانیه.

> ⚠️ **نکته:** مطمئن شوید از منوی بالای صفحه گزینه **Runtime ➔ Change runtime type** روی **T4 GPU** تنظیم شده باشد.

In [ ]:
#@title 🟢 [روش اول] راه‌اندازی ۱۰۰٪ تک‌کلیک با KasmVNC مدرن (Live Real-Time Logs)
#@markdown ### 🌐 تنظیمات آدرس پروژه و رزولوشن صفحه:
#@markdown آدرس سایت، وب‌اپلیکیشن گرافیکی یا دپ مورد نظر خود را جهت لود وارد کنید:
TARGET_URL = "https://webgpureport.org" #@param {type:"string"}
RESOLUTION = "1280x720" #@param ["1280x720", "1366x768", "1600x900", "1920x1080"]

import os, sys, time, subprocess, re, shutil, socket
from IPython.display import display, HTML

def run_step(step_num, title, cmd, show_output=False):
    print(f"\n▶ [{step_num}/6] {title}...", flush=True)
    t0 = time.time()
    if show_output:
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in iter(proc.stdout.readline, ''):
            l = line.strip()
            if l: print(f"  │ {l[:110]}", flush=True)
        proc.wait()
        code = proc.returncode
    else:
        code = subprocess.run(cmd, shell=True).returncode
    dt = time.time() - t0
    status = "✔ انجام شد" if code == 0 else f"✖ خطا (کد {code})"
    print(f"  └─ {status} در {dt:.1f} ثانیه", flush=True)
    return code

# ── 1. سخت‌افزار کرنل، لودر Vulkan و ICD انویدیا ──────────────────────────
run_step(1, "پیکربندی سخت‌افزاری هسته لینوکس، لودر Vulkan و درایور سخت‌افزاری NVIDIA T4", """
mkdir -p /dev/dri /etc/vulkan/icd.d /usr/share/vulkan/icd.d
mknod -m 666 /dev/dri/card0 c 226 0 2>/dev/null || true
mknod -m 666 /dev/dri/renderD128 c 226 128 2>/dev/null || true
mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null || true
chmod -R 666 /dev/dri /dev/nvidia* 2>/dev/null || true

apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libvulkan1 vulkan-tools dbus-x11 >/dev/null 2>&1

cat << 'EOF' > /etc/vulkan/icd.d/nvidia_icd.json
{
  "file_format_version": "1.0.0",
  "ICD": {
    "library_path": "libGLX_nvidia.so.0",
    "api_version": "1.3.275"
  }
}
EOF
cp /etc/vulkan/icd.d/nvidia_icd.json /usr/share/vulkan/icd.d/nvidia_icd.json
echo '/usr/lib64-nvidia' > /etc/ld.so.conf.d/nvidia.conf
ldconfig
/etc/init.d/dbus start >/dev/null 2>&1 || true
""")

# ── 2. کاربر اختصاصی ──────────────────────────────────────────────────────
run_step(2, "ساخت کاربر اختصاصی colab و تنظیم مجوزها", """
rm -f /etc/passwd.lock /etc/shadow.lock /etc/group.lock /etc/gshadow.lock /etc/.pwd.lock
mkdir -p /home/colab
id -u colab >/dev/null 2>&1 || useradd -m -s /bin/bash colab
echo 'colab:123456' | chpasswd
usermod -aG sudo,video,render colab 2>/dev/null || true
echo 'colab ALL=(ALL) NOPASSWD:ALL' > /etc/sudoers.d/colab
id colab
""")

# ── 3. نصب کروم ───────────────────────────────────────────────────────────
if not os.path.exists("/usr/bin/google-chrome"):
    run_step(3, "دانلود و نصب رسمی Google Chrome Stable", """
    wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
    dpkg -i /tmp/chrome.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
    rm -f /tmp/chrome.deb
    """)
else:
    print("\n▶ [3/6] بررسی مرورگر: Google Chrome از قبل نصب است.", flush=True)

# ── 4. نصب KasmVNC 1.5.0 و XFCE4 ─────────────────────────────────────────
res = subprocess.run("lsb_release -cs", shell=True, capture_output=True, text=True)
distro = res.stdout.strip() if res.stdout.strip() in ["jammy", "noble", "focal"] else "jammy"
kasm_deb = f"/tmp/kasmvnc_{distro}.deb"

if not (os.path.exists("/usr/bin/vncserver") and os.path.exists("/usr/share/kasmvnc")):
    run_step(4, f"نصب KasmVNC 1.5.0 ({distro}) و محیط XFCE4", f"""
    DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 xfce4-terminal desktop-base xauth ssl-cert >/dev/null 2>&1
    wget -q -O {kasm_deb} https://github.com/kasmtech/KasmVNC/releases/download/v1.5.0/kasmvncserver_{distro}_1.5.0_amd64.deb
    dpkg -i {kasm_deb} >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
    rm -f {kasm_deb}
    adduser colab ssl-cert >/dev/null 2>&1 || true
    """)
else:
    print("\n▶ [4/6] بررسی KasmVNC: پکیج‌ها از قبل نصب هستند.", flush=True)

# گواهی self-signed: KasmVNC حتی با require_ssl=false مسیر گواهی را load می‌کند و
# گواهی snakeoil پیش‌فرض روی ایمیج کولب وجود ندارد (خطای "certificate file doesn't exist")
os.makedirs("/etc/kasmvnc", exist_ok=True)
subprocess.run(
    "openssl req -x509 -newkey rsa:2048 -nodes -keyout /etc/kasmvnc/kasmvnc.pem "
    "-out /etc/kasmvnc/kasmvnc.pem -days 3650 -subj '/CN=colab' 2>/dev/null; "
    "chmod 644 /etc/kasmvnc/kasmvnc.pem",
    shell=True)

# کانفیگ تمیز kasmvnc.yaml (طبق کانفیگ رفرنس رسمی KasmVNC)
# نکته: فلگ -geometry توسط KasmVNC نادیده گرفته می‌شود؛ رزولوشن را در YAML ست می‌کنیم.
os.makedirs("/home/colab/.vnc", exist_ok=True)
os.makedirs("/etc/kasmvnc", exist_ok=True)
try:
    _rw, _rh = (int(v) for v in RESOLUTION.lower().split("x"))
except Exception:
    _rw, _rh = 1024, 768
kasm_yaml = f"""desktop:
  resolution:
    width: {_rw}
    height: {_rh}

network:
  protocol: http
  interface: 0.0.0.0
  websocket_port: 8443
  use_ipv4: true
  use_ipv6: false
  ssl:
    pem_certificate: /etc/kasmvnc/kasmvnc.pem
    pem_key: /etc/kasmvnc/kasmvnc.pem
    require_ssl: false

command_line:
  prompt: false
"""
with open("/etc/kasmvnc/kasmvnc.yaml", "w") as f: f.write(kasm_yaml)
with open("/home/colab/.vnc/kasmvnc.yaml", "w") as f: f.write(kasm_yaml)

# جلوگیری از دیالوگ‌های مزاحم: keyring ("Choose password for new keyring")
# و First-Run کروم (Terms of Service) با مارکر "First Run"
subprocess.run("rm -f /etc/xdg/autostart/gnome-keyring-*.desktop; "
               "rm -f /usr/share/dbus-1/services/*keyring* "
               "/usr/share/dbus-1/services/org.freedesktop.secrets.service 2>/dev/null; true", shell=True)
os.makedirs("/home/colab/.config/google-chrome", exist_ok=True)
open("/home/colab/.config/google-chrome/First Run", "w").close()

# اجرای خودکار Chrome با پرچم‌های WebGPU از طریق XFCE autostart
# (فلگ -select-de فایل xstartup را بازنویسی می‌کند؛ پس Chrome را از autostart می‌گذاریم)
os.makedirs("/home/colab/.config/autostart", exist_ok=True)
subprocess.run(f"""
cat > /home/colab/launch_chrome.sh << 'CEOF'
#!/bin/bash
export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json
google-chrome-stable --no-sandbox --no-first-run --no-default-browser-check --password-store=basic --disable-gpu-sandbox --enable-unsafe-webgpu --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService --use-angle=vulkan --enable-dawn-features=allow_unsafe_apis,disable_adapter_blocklist --ignore-gpu-blocklist --disable-dev-shm-usage --enable-gpu-rasterization --enable-zero-copy --start-maximized "{TARGET_URL}" &
CEOF
chmod +x /home/colab/launch_chrome.sh
cat > /home/colab/.config/autostart/webgpu-chrome.desktop << 'DEOF'
[Desktop Entry]
Type=Application
Name=WebGPU Google Chrome
Exec=/home/colab/launch_chrome.sh
Terminal=false
DEOF
chown -R colab:colab /home/colab
""", shell=True)

# ثبت پسورد (سوییچ -w ضروری است؛ بدون آن نشست view-only می‌شود و موس/کیبورد کار نمی‌کند)
subprocess.run(
    "printf '%s\\n%s\\n' '123456' '123456' | "
    "sudo -H -u colab env HOME=/home/colab kasmvncpasswd -u colab -w -o >/dev/null 2>&1 || true",
    shell=True)
subprocess.run("chown -R colab:colab /home/colab", shell=True)

# ── 5. تونل Cloudflare ───────────────────────────────────────────────────
if not os.path.exists("/usr/local/bin/cloudflared"):
    run_step(5, "دانلود باینری تونل Cloudflare", """
    wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    chmod +x /usr/local/bin/cloudflared
    """)
else:
    print("\n▶ [5/6] بررسی تونل: Cloudflared آماده است.", flush=True)

# ── 6. استارت سرور VNC و بررسی پورت واقعی ────────────────────────────────
print("\n▶ [6/6] استارت سرور KasmVNC و بازرسی سلامت ارتباط...", flush=True)
subprocess.run("sudo -u colab vncserver -kill :1 >/dev/null 2>&1 || true", shell=True)
subprocess.run("rm -rf /tmp/.X1-lock /tmp/.X11-unix/X1", shell=True)
time.sleep(1)

# فلگ -disableBasicAuth که قبلا استفاده می‌شد در vncserver(1) وجود ندارد و خطای
# unrecognized option می‌داد (KasmVNC احراز هویت را از kasmvnc.yaml می‌خواند).
# بدون -fg اجرا می‌کنیم تا vncserver بعد از daemonize برگردد؛ حلقه بررسی پورت پایین
# تا ۲۰ ثانیه منتظر bind شدن 8443/8444 می‌ماند.
res_vnc = subprocess.run(
    f'sudo -u colab vncserver :1 -geometry {RESOLUTION} -depth 24 -select-de XFCE -disableBasicAuth',
    shell=True, capture_output=True, text=True
)

# تشخیص پورت با socket — ابزارهای ss/netstat روی ایمیج کولب نصب نیستند
vnc_port = None
for _ in range(25):
    time.sleep(1)
    for p in (8443, 8444):
        try:
            s = socket.create_connection(("127.0.0.1", p), timeout=1)
            s.close()
            vnc_port = p
            break
        except OSError:
            continue
    if vnc_port:
        break

if not vnc_port:
    print("❌ خطا: سرور VNC روی پورت 8443/8444 فعال نشد!")
    print("stdout:", res_vnc.stdout or "(خالی)")
    if res_vnc.stderr: print("stderr:", res_vnc.stderr)
    print("\nآخرین خطوط لاگ VNC:")
    subprocess.run("tail -n 25 /home/colab/.vnc/*.log 2>/dev/null", shell=True)
    raise RuntimeError("VNC Server failed to bind to port.")

print(f"  ✔ سرور KasmVNC روی پورت داخلی {vnc_port} با موفقیت مستقر شد.", flush=True)

# استارت/استفاده مجدد از تونل Cloudflare (لینک بین اجراهای مجدد ثابت می‌ماند)
log_file = "/tmp/cloudflared.log"
cloudflared_bin = "/usr/local/bin/cloudflared" if os.path.exists("/usr/local/bin/cloudflared") else (shutil.which("cloudflared") or "cloudflared")

def _tunnel_alive(url):
    import urllib.request, ssl
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    try:
        urllib.request.urlopen(url, timeout=10, context=ctx)
        return True
    except Exception as e:
        return getattr(e, "code", None) in (200, 401, 403)

existing_url = None
if subprocess.run("pgrep -f cloudflared", shell=True, capture_output=True).returncode == 0 and os.path.exists(log_file):
    with open(log_file) as f:
        _m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
    if _m:
        existing_url = _m.group(0)

tunnel_url = None
if existing_url and _tunnel_alive(existing_url):
    tunnel_url = existing_url
    print(f"  ✔ استفاده مجدد از تونل فعال (لینک ثابت): {tunnel_url}", flush=True)
else:
    subprocess.run("pkill -9 -f 'cloudflared' >/dev/null 2>&1 || true", shell=True)
    if os.path.exists(log_file): os.remove(log_file)
    # setsid+nohup: تونل را از چرخه‌ی عمر سلول/شل جدا می‌کنیم تا پایدار بماند
    subprocess.Popen(
        f"setsid nohup {cloudflared_bin} tunnel --no-autoupdate --url http://127.0.0.1:{vnc_port} --logfile {log_file} >/dev/null 2>&1 &",
        shell=True)
    for _ in range(30):
        time.sleep(1)
        if os.path.exists(log_file):
            with open(log_file) as f:
                m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
                if m:
                    tunnel_url = m.group(0)
                    break

if tunnel_url:
    print("\n" + "═" * 65)
    print("🎉 محیط ابری با موفقیت لود شد! (شتاب سخت‌افزاری Tesla T4 فعال است)")
    print(f"🎯 پورت لوکال متصل‌شده: {vnc_port} | مقصد: {TARGET_URL}")
    print("═" * 65)
    banner = f"""
        <div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%); padding: 24px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3); text-align: center; margin: 15px 0;">
            <h2 style="margin: 0 0 10px 0; color: #00e676;">⚡ دسکتاپ KasmVNC آماده اتصال است!</h2>
            <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                ارتباط بدون خطای 502 برقرار شد. برای ورود به محیط روی دکمه زیر کلیک کنید:
            </p>
            <a href="{tunnel_url}" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                👉 ورود به مرورگر ابری (KasmVNC)
            </a>
            <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                لینک مستقیم: <a href="{tunnel_url}" target="_blank" style="color: #64ffda;">{tunnel_url}</a>
            </p>
            <p style="margin-top: 8px; font-size: 12px; opacity: 0.7;">
                بدون نیاز به ورود (احراز هویت غیرفعال است) — فقط روی لینک کلیک کنید.
            </p>
        </div>
    """
    display(HTML(banner))
else:
    print("❌ خطا در ایجاد آدرس تونل Cloudflare.")


In [ ]:
#@title ⚡ [روش دوم] راه‌اندازی با Google Chrome Remote Desktop (Live Real-Time Logs)
#@markdown ### 📋 راهنمای اتصال ریموت دسکتاپ گوگل (تاخیر زیر ۳۰ میلی‌ثانیه و ۶۰ فریم):
#@markdown 1. در یک تب جدید در کامپیوتر خود به **[remotedesktop.google.com/headless](https://remotedesktop.google.com/headless)** بروید.
#@markdown 2. دکمه‌های **Begin** ➔ **Next** ➔ **Authorize** را بزنید و کد لینوکس دبیان را کپی کنید.
#@markdown ⚠️ توجه: کد یک‌بار مصرف است؛ دقیقاً قبل از اجرای سلول آن را دریافت و پیست کنید.
AUTH_COMMAND = "" #@param {type:"string"}
PIN = 123456 #@param {type:"integer"}
TARGET_URL = "https://webgpureport.org" #@param {type:"string"}

import os, re, time, subprocess
from IPython.display import display, HTML

# پین باید همیشه ۶ رقمی رشته‌ای باشد
PIN = str(PIN).zfill(6)[:6]

raw_auth = AUTH_COMMAND.strip()
if not raw_auth:
    print("⚠️ فیلد AUTH_COMMAND خالی است!")
    print("👈 دریافت کد لینوکس: https://remotedesktop.google.com/headless")
    try:
        raw_auth = input("دستور احراز هویت را وارد کنید: ").strip()
    except (EOFError, OSError):
        raw_auth = ""

if not raw_auth:
    raise ValueError("دستور احراز هویت الزامی است.")

def run_step(step_num, title, cmd):
    print(f"\n▶ [{step_num}/5] {title}...", flush=True)
    t0 = time.time()
    code = subprocess.run(cmd, shell=True).returncode
    dt = time.time() - t0
    status = "✔ انجام شد" if code == 0 else f"✖ خطا (کد {code})"
    print(f"  └─ {status} در {dt:.1f} ثانیه", flush=True)
    return code

# ── 1. درایورهای گرافیک و Vulkan ─────────────────────────────────────────
run_step(1, "پیکربندی سخت‌افزاری هسته لینوکس، لودر Vulkan و درایور تسلا T4", """
mkdir -p /dev/dri /etc/vulkan/icd.d /usr/share/vulkan/icd.d
mknod -m 666 /dev/dri/card0 c 226 0 2>/dev/null || true
mknod -m 666 /dev/dri/renderD128 c 226 128 2>/dev/null || true
mknod -m 666 /dev/nvidia-modeset c 195 254 2>/dev/null || true
chmod -R 666 /dev/dri /dev/nvidia* 2>/dev/null || true

apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libvulkan1 vulkan-tools dbus-x11 >/dev/null 2>&1

cat << 'EOF' > /etc/vulkan/icd.d/nvidia_icd.json
{
  "file_format_version": "1.0.0",
  "ICD": {
    "library_path": "libGLX_nvidia.so.0",
    "api_version": "1.3.275"
  }
}
EOF
cp /etc/vulkan/icd.d/nvidia_icd.json /usr/share/vulkan/icd.d/nvidia_icd.json
echo '/usr/lib64-nvidia' > /etc/ld.so.conf.d/nvidia.conf
ldconfig
/etc/init.d/dbus start >/dev/null 2>&1 || true
""")

# ── 2. کاربر اختصاصی ─────────────────────────────────────────────────────
run_step(2, "ساخت کاربر اختصاصی colab و دسترسی‌های سیستمی", """
mkdir -p /home/colab
id -u colab >/dev/null 2>&1 || useradd -m -s /bin/bash colab
echo 'colab:123456' | chpasswd
usermod -aG sudo,video,render colab 2>/dev/null || true
echo 'colab ALL=(ALL) NOPASSWD:ALL' > /etc/sudoers.d/colab
""")

# ── 3. پکیج‌های XFCE و Chrome Remote Desktop ──────────────────────────────
run_step(3, "نصب پکیج‌های دسکتاپ XFCE4 و Chrome Remote Desktop", """
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq xfce4 desktop-base xfce4-terminal >/dev/null 2>&1
DEBIAN_FRONTEND=noninteractive apt-get purge -y -qq xscreensaver >/dev/null 2>&1 || true
if [ ! -f /opt/google/chrome-remote-desktop/start-host ]; then
  wget -q -O /tmp/crd.deb https://dl.google.com/linux/direct/chrome-remote-desktop_current_amd64.deb
  dpkg -i /tmp/crd.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
  rm -f /tmp/crd.deb
fi
usermod -aG chrome-remote-desktop colab 2>/dev/null || true

cat << 'EOF' > /home/colab/.chrome-remote-desktop-session
export DESKTOP_SESSION=xfce
export XDG_CURRENT_DESKTOP=XFCE
export XDG_CONFIG_DIRS=/etc/xdg/xdg-xfce:/etc/xdg
exec /usr/bin/xfce4-session
EOF
cp /home/colab/.chrome-remote-desktop-session /etc/chrome-remote-desktop-session
chmod +x /home/colab/.chrome-remote-desktop-session
""")

# ── 4. کروم و اسکریپت WebGPU در Startup دسکتاپ ───────────────────────────
run_step(4, "نصب Google Chrome و تنظیم اجرای خودکار با شتاب Dawn WebGPU", f"""
if [ ! -f /usr/bin/google-chrome ]; then
  wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
  dpkg -i /tmp/chrome.deb >/dev/null 2>&1 || apt-get install -fy -qq >/dev/null 2>&1
  rm -f /tmp/chrome.deb
fi
rm -f /etc/xdg/autostart/gnome-keyring-*.desktop
rm -f /usr/share/dbus-1/services/*keyring* /usr/share/dbus-1/services/org.freedesktop.secrets.service 2>/dev/null || true
mkdir -p /home/colab/.config/autostart /home/colab/Desktop /home/colab/.config/google-chrome
touch "/home/colab/.config/google-chrome/First Run"
cat << 'EOF' > /home/colab/launch_chrome.sh
#!/bin/bash
export VK_ICD_FILENAMES=/etc/vulkan/icd.d/nvidia_icd.json
google-chrome-stable \\
  --no-first-run \\
  --no-default-browser-check \\
  --password-store=basic \\
  --no-sandbox \\
  --disable-gpu-sandbox \\
  --enable-unsafe-webgpu \\
  --enable-features=Vulkan,DefaultANGLEVulkan,VulkanFromANGLE,WebGPUService \\
  --use-angle=vulkan \\
  --enable-dawn-features=allow_unsafe_apis,disable_adapter_blocklist \\
  --disable-dawn-features=disallow_unsafe_apis \\
  --ignore-gpu-blocklist \\
  --disable-dev-shm-usage \\
  --enable-gpu-rasterization \\
  --enable-zero-copy \\
  --start-maximized \\
  "{TARGET_URL}" &
EOF
chmod +x /home/colab/launch_chrome.sh
cat << 'EOF' > /home/colab/.config/autostart/webgpu-chrome.desktop
[Desktop Entry]
Version=1.0
Type=Application
Name=WebGPU Google Chrome
Exec=/home/colab/launch_chrome.sh
Icon=google-chrome
Terminal=false
EOF
cp /home/colab/.config/autostart/webgpu-chrome.desktop /home/colab/Desktop/
chmod +x /home/colab/Desktop/webgpu-chrome.desktop
chown -R colab:colab /home/colab
""")

# ── 5. استارت هاست با ثبت پین از طریق stdin ──────────────────────────────
print("\n▶ [5/5] ثبت و راه‌اندازی هاست ریموت دسکتاپ گوگل با اکانت شما...", flush=True)

# پاک‌سازی دستور گوگل:
#  - پیشوند DISPLAY= حذف می‌شود (در نسخه‌های جدید CRD باعث unrecognized argument می‌شد)
#  - --pin=... حذف؛ پین از طریق stdin تغذیه می‌شود
clean_cmd = re.sub(r'^DISPLAY=\s*', '', raw_auth)
clean_cmd = re.sub(r'--pin=\S+', '', clean_cmd).strip()

runner_script = f"""#!/bin/bash
printf '{PIN}\\n{PIN}\\n' | {clean_cmd}
sleep 2
/opt/google/chrome-remote-desktop/chrome-remote-desktop --start || true
"""
with open("/tmp/start_crd.sh", "w", encoding="utf-8") as f:
    f.write(runner_script)
subprocess.run("chmod 755 /tmp/start_crd.sh && chown colab:colab /tmp/start_crd.sh", shell=True)

res = subprocess.run("su - colab -s /bin/bash -c /tmp/start_crd.sh",
                     shell=True, capture_output=True, text=True)
time.sleep(3)
check = subprocess.run("ps aux | grep -v grep | grep chrome-remote-desktop",
                       shell=True, capture_output=True, text=True)

print("\n" + "═" * 65)
if "chrome-remote-desktop" in check.stdout:
    print("🎉 سرویس Google Chrome Remote Desktop با موفقیت استارت شد!")
    print(f"🔑 پین اتصال: {PIN}")
    print("═" * 65)
    banner = f"""
        <div style="background: linear-gradient(135deg, #0d47a1 0%, #1976d2 100%); padding: 24px; border-radius: 12px; color: #fff; font-family: sans-serif; box-shadow: 0 4px 20px rgba(0,0,0,0.3); text-align: center; margin: 15px 0;">
            <h2 style="margin: 0 0 10px 0; color: #64ffda;">🚀 دسکتاپ ابری ۶۰ فریم آماده اتصال است!</h2>
            <p style="font-size: 15px; margin-bottom: 20px; opacity: 0.95;">
                دستگاه در پنل ریموت گوگل ثبت شد. روی دکمه زیر کلیک کرده و پین <b>{PIN}</b> را وارد کنید:
            </p>
            <a href="https://remotedesktop.google.com/access" target="_blank" style="padding: 14px 34px; background: #00e676; color: #000; font-weight: bold; font-size: 18px; text-decoration: none; border-radius: 8px; display: inline-block; box-shadow: 0 4px 15px rgba(0,230,118,0.4);">
                👉 ورود به Chrome Remote Desktop
            </a>
            <p style="margin-top: 15px; font-size: 13px; opacity: 0.8;">
                آدرس مستقیم: <a href="https://remotedesktop.google.com/access" target="_blank" style="color: #64ffda;">https://remotedesktop.google.com/access</a>
            </p>
        </div>
    """
    display(HTML(banner))
else:
    print("⚠️ خروجی فراخوانی:")
    print(res.stdout or "(خالی)")
    if res.stderr: print("خطاها:", res.stderr)
    print("═" * 65)
    print("نکته مهم: کد تأیید گوگل یک‌بار مصرف است. مطمئن شوید دقیقاً قبل از اجرای سلول آن را از remotedesktop.google.com/headless کپی کرده‌اید.")


In [ ]:
#@title 📊 مانیتورینگ زنده کارت گرافیک و توان پردازشی تسلا T4 (Universal Hardware GPU Monitor)
#@markdown این سلول وضعیت زنده درصد لود پردازشی GPU، حافظه VRAM، دما و پروسه‌های فعال کارت گرافیک را نمایش می‌دهد:

import time, subprocess, shutil
from IPython.display import clear_output

if not shutil.which("nvidia-smi"):
    print("❌ خطا: کارت گرافیک شناسایی نشد!")
    print("👈 لطفاً از منوی بالای کولب مسیر زیر را دنبال کرده و نوت‌بوک را روی T4 GPU قرار دهید:")
    print("   Runtime ➔ Change runtime type ➔ T4 GPU ➔ Save")
else:
    print("🚀 مانیتورینگ زنده توان پردازشی تسلا T4 در حال اجراست (برای توقف، دکمه Stop سلول را بزنید)...")
    try:
        while True:
            res = subprocess.run([
                "nvidia-smi",
                "--query-gpu=name,utilization.gpu,utilization.memory,memory.used,memory.total,temperature.gpu",
                "--format=csv,noheader,nounits"
            ], capture_output=True, text=True)

            if res.returncode == 0 and res.stdout.strip():
                parts = [p.strip() for p in res.stdout.strip().split(",")]
                if len(parts) >= 6:
                    name, gpu_util, mem_util, mem_used, mem_total, temp = parts[:6]
                    clear_output(wait=True)
                    print("=" * 65)
                    print(f"🚀 UNIVERSAL WEBGPU COMPUTE & GRAPHICS MONITOR — {name}")
                    print("=" * 65)
                    print(f"  ⚡ GPU Compute Utilization : {gpu_util}%")
                    print(f"  🧠 VRAM Memory Used        : {mem_used} MiB / {mem_total} MiB ({mem_util}%)")
                    print(f"  🌡️  GPU Temperature         : {temp} °C")
                    print("=" * 65)
                    print("  💡 برای متوقف کردن مانیتور، دکمه Stop (■) سلول را بزنید.")
            time.sleep(1)
    except KeyboardInterrupt:
        print("\nمانیتور با موفقیت متوقف شد.")
